In [1]:
# ============================================================
# Construcción de matrices gamma (Clifford) en d=4 y d=6
# usando SymPy, y verificación de anticonmutadores.
#
# Convención: Euclídea -> {Gamma^A, Gamma^B} = 2 δ^{AB} I
# ============================================================

import sympy as sp

I = sp.I

# -----------------------------
# 1) Matrices de Pauli e identidades
# -----------------------------
sigma1 = sp.Matrix([[0, 1],
                    [1, 0]])

sigma2 = sp.Matrix([[0, -I],
                    [I,  0]])

sigma3 = sp.Matrix([[1,  0],
                    [0, -1]])

I2 = sp.eye(2)

def kron(A, B):
    """Producto de Kronecker (tensorial) SymPy."""
    return sp.kronecker_product(A, B)

def anticomm(A, B):
    """Anticonmutador {A,B} = AB + BA."""
    return A*B + B*A

def is_zero_matrix(M):
    """Chequea si una matriz es exactamente la matriz cero (simbólica)."""
    return M.equals(sp.zeros(*M.shape))

def check_clifford(gammas, label=""):
    """
    Verifica {Γ^A, Γ^B} = 2 δ^{AB} I_n para un conjunto de gammas.
    Imprime resultados y devuelve True/False.
    """
    n = gammas[0].shape[0]
    Id = sp.eye(n)
    ok = True
    d = len(gammas)

    print("\n" + "="*60)
    print(f"Chequeo de Clifford {label}: d={d}, tamaño={n}x{n}")
    print("Objetivo: {Γ^A, Γ^B} = 2 δ^{AB} I")
    print("="*60)

    # Chequeo completo
    for a in range(d):
        for b in range(d):
            lhs = sp.simplify(anticomm(gammas[a], gammas[b]))
            rhs = 2*Id if a == b else sp.zeros(n)
            if not lhs.equals(rhs):
                ok = False
                # Muestra un resumen del fallo
                diff = sp.simplify(lhs - rhs)
                print(f"FALLA en (A,B)=({a+1},{b+1}). Máximo |diff| simbólico no aplica.")
                print("LHS - RHS =")
                sp.pprint(diff)
                print("-"*60)
                # si quieres cortar en el primer fallo, descomenta:
                # return False

    if ok:
        print("✅ Todo OK: se cumple la álgebra de Clifford.")
    else:
        print("❌ Hay fallos (ver arriba). Revisa la construcción/convención.")

    return ok


# ============================================================
# 2) Construcción en d = 4 (euclídeo)  -> matrices 4x4
# ============================================================
# gamma^1 = σ1 ⊗ σ1
# gamma^2 = σ1 ⊗ σ2
# gamma^3 = σ1 ⊗ σ3
# gamma^4 = σ2 ⊗ I2
gamma4 = [
    kron(sigma1, sigma1),
    kron(sigma1, sigma2),
    kron(sigma1, sigma3),
    kron(sigma2, I2),
]

print("Gammas d=4 (euclídeo):")
for i, g in enumerate(gamma4, start=1):
    print(f"\nγ^{i} =")
    sp.pprint(g)

check_clifford(gamma4, label="(d=4 euclídeo)")


# ============================================================
# 3) Construcción en d = 6 (euclídeo) -> matrices 8x8
# ============================================================
# Γ^1 = σ1 ⊗ I ⊗ I
# Γ^2 = σ2 ⊗ I ⊗ I
# Γ^3 = σ3 ⊗ σ1 ⊗ I
# Γ^4 = σ3 ⊗ σ2 ⊗ I
# Γ^5 = σ3 ⊗ σ3 ⊗ σ1
# Γ^6 = σ3 ⊗ σ3 ⊗ σ2
Gamma6 = [
    kron(sigma1, kron(I2, I2)),
    kron(sigma2, kron(I2, I2)),
    kron(sigma3, kron(sigma1, I2)),
    kron(sigma3, kron(sigma2, I2)),
    kron(sigma3, kron(sigma3, sigma1)),
    kron(sigma3, kron(sigma3, sigma2)),
]

print("\n\nGammas d=6 (euclídeo):")
for i, G in enumerate(Gamma6, start=1):
    print(f"\nΓ^{i} =")
    sp.pprint(G)

check_clifford(Gamma6, label="(d=6 euclídeo)")


# ============================================================
# 4) Verificaciones adicionales útiles
# ============================================================

# (a) Chequear cuadrados
def check_squares(gammas, name="Γ"):
    n = gammas[0].shape[0]
    Id = sp.eye(n)
    ok = True
    for i, G in enumerate(gammas, start=1):
        if not (sp.simplify(G*G).equals(Id)):
            ok = False
            print(f"❌ {name}^{i} no cumple ({name}^{i})^2 = I")
        else:
            print(f"✅ ({name}^{i})^2 = I")
    return ok

print("\nChequeo de cuadrados d=4:")
check_squares(gamma4, name="γ")

print("\nChequeo de cuadrados d=6:")
check_squares(Gamma6, name="Γ")


# (b) Construir la matriz de quiralidad en d=6: Γ^7 = i^3 Γ^1...Γ^6 = -i Γ^1...Γ^6
Gamma7 = (I**3)
prod = sp.eye(8)
for G in Gamma6:
    prod = sp.simplify(prod*G)
Gamma7 = sp.simplify(Gamma7 * prod)

print("\nΓ^7 (d=6):")
sp.pprint(Gamma7)

# Verificar (Γ^7)^2 = I y {Γ^7, Γ^A}=0
print("\nChequeo Γ^7:")
print(" (Γ^7)^2 = I ? ->", sp.simplify(Gamma7*Gamma7).equals(sp.eye(8)))

for a, G in enumerate(Gamma6, start=1):
    test = sp.simplify(anticomm(Gamma7, G))
    print(f" {{Γ^7, Γ^{a}}} = 0 ? ->", test.equals(sp.zeros(8)))


# ============================================================
# 5) (Opcional) Pasar a Minkowski desde Euclídeo (ejemplo)
# ============================================================
# Si quieres firma (+-----) en 6D:
#  - toma Γ^0_M = Γ^1_E (por ejemplo) y para espaciales Γ^i_M = i Γ^i_E
# Esto depende de qué índice elijas como "tiempo".
#
# Aquí solo dejo una plantilla:
#
# GammaM = [Gamma6[0]] + [I*G for G in Gamma6[1:]]
# y entonces {Γ^μ, Γ^ν} = 2 η^{μν} I con η=diag(+,-,-,-,-,-)
# ============================================================

Gammas d=4 (euclídeo):

γ^1 =
⎡0  0  0  1⎤
⎢          ⎥
⎢0  0  1  0⎥
⎢          ⎥
⎢0  1  0  0⎥
⎢          ⎥
⎣1  0  0  0⎦

γ^2 =
⎡0  0   0  -ⅈ⎤
⎢            ⎥
⎢0  0   ⅈ  0 ⎥
⎢            ⎥
⎢0  -ⅈ  0  0 ⎥
⎢            ⎥
⎣ⅈ  0   0  0 ⎦

γ^3 =
⎡0  0   1  0 ⎤
⎢            ⎥
⎢0  0   0  -1⎥
⎢            ⎥
⎢1  0   0  0 ⎥
⎢            ⎥
⎣0  -1  0  0 ⎦

γ^4 =
⎡0  0  -ⅈ  0 ⎤
⎢            ⎥
⎢0  0  0   -ⅈ⎥
⎢            ⎥
⎢ⅈ  0  0   0 ⎥
⎢            ⎥
⎣0  ⅈ  0   0 ⎦

Chequeo de Clifford (d=4 euclídeo): d=4, tamaño=4x4
Objetivo: {Γ^A, Γ^B} = 2 δ^{AB} I
✅ Todo OK: se cumple la álgebra de Clifford.


Gammas d=6 (euclídeo):

Γ^1 =
⎡0  0  0  0  1  0  0  0⎤
⎢                      ⎥
⎢0  0  0  0  0  1  0  0⎥
⎢                      ⎥
⎢0  0  0  0  0  0  1  0⎥
⎢                      ⎥
⎢0  0  0  0  0  0  0  1⎥
⎢                      ⎥
⎢1  0  0  0  0  0  0  0⎥
⎢                      ⎥
⎢0  1  0  0  0  0  0  0⎥
⎢                      ⎥
⎢0  0  1  0  0  0  0  0⎥
⎢                      ⎥
⎣0  0  0  1  0  0  0  0⎦

Γ^2